In [ ]:
%pip install uv --quiet
%uv pip install pandas numpy plotly matplotlib scipy
%uv sync

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.2 environment at: c:\Users\cbutt\OneDrive\Desktop\DataScience\Project\Modern-Store-of-Value\.venv
Checked 6 packages in 26ms


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

In [ ]:
stooq_tickers = {
    "Crypto ETFs": ["BITW", "IBIT", "ETHA"], 
    "Individual Stocks": ["NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"],
    "Sector ETFs": ["XLU"],
    "Broad Market ETFs": ["SPY", "VTI"],
    "Commodity ETFs (Metals)": ["GLD", "SLV", "PPLT", "PALL"],
    "Commodity ETFs (Agriculture)": ["WEAT", "SOYB", "DBA"]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [26]:
# Create flat mapping for Category
category_map = {ticker: cat for cat, ticks in stooq_tickers.items() for ticker in ticks}

with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    data = processor.download(stooq_tickers, start=start_date, end=end_date)

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map.get(ticker, "Other"))

combined_data = pd.concat(data.values()).reset_index()
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

In [4]:
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    category_map = processor.build_category_map(stooq_tickers)
    data = processor.download(
        stooq_tickers,
        start=start_date,
        end=end_date,
    )

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map[ticker])

combined_data = pd.concat(data.values()).reset_index()

# combined_data.head()
data["AAPL"].tail()

,Open,High,Low,Close,Volume,OpenInt,Ticker,Category
Date,,,,,,,,
2025-11-30,270.158,280.380,265.32,278.85,877813393,0,AAPL,Individual Stocks
2025-12-31,278.010,288.620,266.95,271.86,924529010,0,AAPL,Individual Stocks
2026-01-31,272.255,277.840,243.42,259.48,1040017271,0,AAPL,Individual Stocks
2026-02-28,260.030,280.905,255.45,264.18,988633102,0,AAPL,Individual Stocks
2026-03-26,262.410,266.530,246.00,252.89,763091456,0,AAPL,Individual Stocks


# Functions

In [27]:
def evaluate_crisis_performance(df_long):
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    # Widened windows to ensure 2+ monthly points are captured
    crisis_events = {
        "2022 Bear Market":    ("2021-12-31", "2022-10-31"),
        "2023 Banking Crisis": ("2023-02-28", "2023-05-31"),
        "2025 Tariff Shock":   ("2025-03-31", "2025-04-30"),
        "2026 Iran War":       ("2026-02-28", "2026-03-31")
    }

    results = []
    for name, (start, end) in crisis_events.items():
        window = pivot_df.loc[start:end]
        if len(window) < 2: continue
            
        for ticker in window.columns:
            series = window[ticker].dropna()
            if len(series) >= 2:
                ret = (series.iloc[-1] / series.iloc[0]) - 1
                results.append({'Crisis': name, 'Ticker': ticker, 'Return': ret * 100, 'Category': category_map[ticker]})

    res_df = pd.DataFrame(results)
    gold_rets = res_df[res_df['Ticker'] == 'GLD'].set_index('Crisis')['Return']
    res_df['Excess vs Gold (%)'] = res_df.apply(lambda x: x['Return'] - gold_rets.get(x['Crisis'], 0), axis=1)
    res_df['Success'] = np.where(res_df['Excess vs Gold (%)'] > 0, "YES", "NO")

    # Faceted bar chart matching Logan's style
    fig = px.bar(res_df, x="Ticker", y="Excess vs Gold (%)", color="Category", 
                 facet_col="Crisis", facet_col_wrap=2, template="plotly_white",
                 title="Success Metric: Excess Returns vs. Gold During Crisis Windows")
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()
    return res_df

In [30]:
def calculate_resilience_recovery(df_long):
    """
    Calculates the average recovery time for every asset after a 5% drop.
    Success = Faster recovery than Gold.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    recovery_stats = []
    for ticker in pivot_df.columns:
        prices = pivot_df[ticker].dropna()
        rolling_max = prices.cummax()
        drawdown = (prices - rolling_max) / rolling_max
        
        # Calculate how many weeks it stays below peak
        is_underwater = drawdown < 0
        # This is a simplified proxy for 'average recovery time'
        underwater_weeks = is_underwater.sum() 
        
        recovery_stats.append({'Ticker': ticker, 'Total Weeks Underwater': underwater_weeks})
        
    res_df = pd.DataFrame(recovery_stats).sort_values('Total Weeks Underwater')
    
    fig = px.bar(res_df, x='Ticker', y='Total Weeks Underwater', 
                 title="Resilience: Total Weeks Spent Below Previous Peak",
                 template="plotly_white")
    fig.show()

In [31]:
def discover_market_shocks(df_long, target_ticker='SPY', sigma_threshold=2):
    """
    Finds dates where the market drop was more than 2 Standard Deviations from normal.
    Use these dates to find 'Crisis Events' for your report.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    rets = pivot_df[target_ticker].pct_change()
    
    mean = rets.mean()
    std = rets.std()
    
    # A 'Shock' is a drop worse than (Mean - 2*StdDev)
    shocks = rets[rets < (mean - sigma_threshold * std)]
    
    print(f"--- Systemic Shocks detected for {target_ticker} ---")
    for date, val in shocks.items():
        print(f"Shock Date: {date.date()} | Drop: {val*100:.2f}%")
        
    return shocks

In [32]:
evaluate_crisis_performance(combined_data)

,Crisis,Ticker,Return,Category,Excess vs Gold (%),Success
0,2022 Bear Market,AAPL,-13.286146,Individual Stocks,-2.143189,NO
1,2022 Bear Market,AMD,-58.262682,Individual Stocks,-47.119725,NO
2,2022 Bear Market,AMZN,-38.554557,Individual Stocks,-27.411599,NO
3,2022 Bear Market,DBA,0.405063,Commodity ETFs (Agriculture),11.548021,YES
4,2022 Bear Market,GLD,-11.142957,Commodity ETFs (Metals),0.000000,NO
...,...,...,...,...,...,...
80,2026 Iran War,TSLA,-7.552607,Individual Stocks,9.627754,YES
81,2026 Iran War,VTI,-5.673466,Broad Market ETFs,11.506896,YES
82,2026 Iran War,WEAT,2.436863,Commodity ETFs (Agriculture),19.617225,YES
83,2026 Iran War,WMT,-4.509574,Individual Stocks,12.670788,YES


In [34]:
calculate_resilience_recovery(combined_data)

In [35]:
discover_market_shocks(combined_data)

--- Systemic Shocks detected for SPY ---
Shock Date: 2022-04-30 | Drop: -8.78%
Shock Date: 2022-06-30 | Drop: -8.25%
Shock Date: 2022-09-30 | Drop: -9.24%


Date
2022-04-30   -0.087770
2022-06-30   -0.082454
2022-09-30   -0.092441
Name: SPY, dtype: float64